## Julieta Madrigal Flores - 744029
## Lunes 11 de mayo del 2026

Investiga los siguientes conceptos y su conexión con las particiones en árboles de clasificación:

- GINI
- Entropía
- Log Loss
- ¿Cuál es la diferencia entre entropía y log loss?

Escoge un dataset y realiza el ejemplo de hacer una partición para cada uno de los criterios de decisión.

# **Teoría**

# Criterios de Partición en Árboles de Clasificación

En los árboles de **regresión**, el criterio de partición minimiza la **varianza conjunta** (MSE) en cada split. En clasificación, las salidas son categorías, por lo que no existe una varianza calculable directamente. En su lugar, se utilizan métricas de **impureza** o **incertidumbre** del nodo: mientras más mezcladas estén las clases = peor es la partición.

## 1. Índice de Gini

El índice de Gini mide la probabilidad de clasificar incorrectamente un elemento si se le asigna una etiqueta al azar según la distribución del nodo:

$$Gini = 1 - \sum_{k=1}^{K} p_k^2$$

- **Rango:** [0, 0.5] para clasificación binaria (0 = nodo puro, 0.5 = máxima impureza)
- **Ventaja:** No requiere cálculo de logaritmos, por lo que es más rápido computacionalmente
- **Comportamiento:** Tiende a aislar la clase más frecuente en una rama
- **Uso en sklearn:** Es el criterio por defecto (criterion='gini')

Para evaluar un split, se calcula el Gini **ponderado** de los nodos hijos:

$$Gini_{split} = \frac{n_L}{n} \cdot Gini_L + \frac{n_R}{n} \cdot Gini_R$$

Se elige el split que **minimiza** este valor.

## 2. Entropía (Information Gain)

Proviene de la teoría de información de Shannon (1948), mide cuántos bits son necesarios para describir la distribución de clases en un nodo:

$$H = -\sum_{k=1}^{K} p_k \log_2(p_k)$$

- **Rango:** [0, log₂(K)] para 2 clases: [0, 1]
- **Comportamiento:** Penaliza más a las impurezas extremas en comparación con Gini
- **Uso en sklearn:** criterion='entropy'

La **Ganancia de Información** (Information Gain) al hacer un split es:

$$IG = H(\text{padre}) - \left(\frac{n_L}{n} \cdot H_L + \frac{n_R}{n} \cdot H_R\right)$$

Se elige el split que **maximiza** esta ganancia.


## 3. Log Loss (Cross-Entropy Loss)

$$LogLoss = -\frac{1}{N}\sum_{i=1}^{N}\sum_{k=1}^{K} y_{ik} \log(\hat{p}_{ik})$$

- Evalúa qué tan bien el modelo **calibra sus probabilidades predichas** vs. las etiquetas reales
- Penaliza duramente las predicciones **confiadas pero incorrectas**
- **Uso en sklearn:** criterion='log_loss'

## 4. ¿Entropía y Log Loss son diferentes?

La entropía habla de un nodo y el log loss del dataset completo; sin embargo, **matemáticamente son equivalentes**. Dentro de cada hoja $m$ del árbol, la predicción es **constante**: todos los puntos que caen en esa hoja reciben la misma probabilidad $p_{mk}$ (la proporción observada de cada clase).

Ahora bien, si se expande el Log Loss del árbol completo:

$$LogLoss = -\frac{1}{N}\sum_{\text{hojas } m} \sum_{i \in m} \sum_k y_{ik} \log(p_{mk})$$

Como $p_{mk}$ es constante dentro de la hoja, la suma interna colapsa en:

$$= \frac{1}{N}\sum_{\text{hojas } m} n_m \cdot H_m$$

El `criterion='entropy'` y `criterion='log_loss'` en sklearn **producen exactamente el mismo árbol**. Son dos nombres para el mismo criterio matemático, expresados desde perspectivas distintas.

La única diferencia técnica es la **base del logaritmo**: entropía usa base 2 (resultado en bits) y log loss usa base $e$ (resultado en nats). Esto los escala diferente pero no cambia cuál split se elige porque multiplicar por una constante ($\ln 2$) no altera el orden de los candidatos.





# **Código**

In [14]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score
from sklearn.preprocessing import LabelEncoder

In [15]:
data = sns.load_dataset('titanic')
data = data[['survived', 'pclass', 'sex', 'age', 'fare']].dropna()
data.head()

,survived,pclass,sex,age,fare
0,0,3,male,22.0,7.2500
1,1,1,female,38.0,71.2833
2,1,3,female,26.0,7.9250
3,1,1,female,35.0,53.1000
4,0,3,male,35.0,8.0500


In [16]:
data.tail()

,survived,pclass,sex,age,fare
885,0,3,female,39.0,29.125
886,0,2,male,27.0,13.000
887,1,1,female,19.0,30.000
889,1,1,male,26.0,30.000
890,0,3,male,32.0,7.750


In [17]:
data.describe()

,survived,pclass,age,fare
count,714.000000,714.000000,714.000000,714.000000
mean,0.406162,2.236695,29.699118,34.694514
std,0.491460,0.838250,14.526497,52.918930
min,0.000000,1.000000,0.420000,0.000000
25%,0.000000,1.000000,20.125000,8.050000
50%,0.000000,2.000000,28.000000,15.741700
75%,1.000000,3.000000,38.000000,33.375000
max,1.000000,3.000000,80.000000,512.329200


In [18]:
le = LabelEncoder()
data['sex'] = le.fit_transform(data['sex'])  # female=0, male=1

# Definir variables
X = data[['pclass', 'sex', 'age', 'fare']]
y = data['survived']

In [19]:
# Hacer train - test split
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

In [20]:
## FUNCIONES

# 1. GINI
# Cuanto más cerca esté el valor de cero, más puro es el nodo donde todas las muestras pertenecen a la misma clase
# cuanto más cerca esté de uno, más impuro es = las muestras están igualmente distribuidas en varias clases
# np.bincount(y) cuenta cuántos hay de cada clase y al dividir entre len(y) obtienes las proporciones p_k
# Luego aplica la fórmula 1
# El if len(y) == 0 evita división entre cero si un nodo queda vacío

# Se divide entre total para obtener proporciones.
# Gini = 0 si todas las muestras son misma clase (puro)
# Gini = 0.5 para clases balanceadas 50%-50%
def gini(y):
    if len(y) == 0:
        return 0
    p = np.bincount(y) / len(y)
    return 1 - np.sum(p**2)

# Un nodo es puro si contiene únicamente ejemplos de una sola clase
# calculan la impureza ponderada después del split
# se pondera por el tamaño de cada rama
def gini_split(y_left, y_right):
    n = len(y_left) + len(y_right)
    return (len(y_left)/n) * gini(y_left) + (len(y_right)/n) * gini(y_right)

# 2. ENTROPÍA
# la suma de las proporciones de cada clase multiplicada por el logaritmo de esas proporciones
# p = p[p > 0] es importante: si una clase tiene probabilidad 0 =log2(0) es infinito negativo y rompe el cálculo
# se filtran los ceros se evitar ese problema
def entropy(y):
    if len(y) == 0:
        return 0
    p = np.bincount(y) / len(y)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

# Un nodo es puro si contiene únicamente ejemplos de una sola clase
# calculan la impureza ponderada después del split
# se pondera por el tamaño de cada rama
def entropy_split(y_left, y_right):
    n = len(y_left) + len(y_right)
    return (len(y_left)/n) * entropy(y_left) + (len(y_right)/n) * entropy(y_right)

# La ganancia de información es = cuánta entropía se redujo al hacer el split
# Si el nodo padre tenía 0.96 bits y los hijos en promedio tienen 0.74 bits = 0.22 bits de información
# IG = Entropía padre - Entropía ponderada hijos
def information_gain(y_parent, y_left, y_right):
    return entropy(y_parent) - entropy_split(y_left, y_right)

In [21]:
#  PARTICIÓN MANUAL POR FEATURE: sex, osea simula un solo split en el nodo raíz usando la variable sex
# Se extraen los valores de y para los participantes de cada rama.
# .values convierte de pandas Series a numpy array, que es lo que esperan las funciones definidas antes.
y_arr   = y_train.values
sex_arr = X_train['sex'].values

# y_left  = etiquetas de supervivencia de las mujeres (sex=0)
# y_right = etiquetas de supervivencia de los hombres (sex=1)
y_left  = y_arr[sex_arr == 0]
y_right = y_arr[sex_arr == 1]

In [22]:
# GINI
#   Mayor reducción = mejor split
# Calcula el Gini antes y después del split
# La diferencia es la reducción de Gini: qué tanto bajó la impureza al partir por sexo
g_padre = gini(y_arr)
g_split = gini_split(y_left, y_right)
gini_gain = g_padre - g_split

print(f"   Gini padre        = {g_padre:.4f}")
print(f"   Gini izq (F)      = {gini(y_left):.4f}")
print(f"   Gini der (M)      = {gini(y_right):.4f}")
print(f"   Gini ponderado    = {g_split:.4f}")
print(f"   Reducción Gini    = {gini_gain:.4f}  mayor es mejor")

   Gini padre        = 0.4837
   Gini izq (F)      = 0.3457
   Gini der (M)      = 0.3207
   Gini ponderado    = 0.3297
   Reducción Gini    = 0.1540  mayor es mejor


El nodo principal tenía un Gini de 0.4837, esto indica que las clases estaban casi perfectamente mezcladas: mucha incertidumbre sobre si un pasajero sobreviviría o no. Después de partir por sexo, el Gini ponderado bajó a 0.3297, esto confirma que sex es un feature muy discriminativo: las mujeres y los hombres tienen distribuciones de supervivencia muy distintas, por lo que la partición genera nodos mucho más puros.

In [23]:
# ENTROPÍA
# Information Gain = H(padre) - entropía ponderada de hijos.
#  Se mide en bits (base 2).
#  Mayor IG = mejor split
h_padre = entropy(y_arr)
ig = information_gain(y_arr, y_left, y_right) #  ganancia de información en bits
h_pond  = entropy_split(y_left, y_right)

print(f"   Entropía padre    = {h_padre:.4f} bits")
print(f"   Entropía izq (F)  = {entropy(y_left):.4f} bits")
print(f"   Entropía der (M)  = {entropy(y_right):.4f} bits")
print(f"   Entropía ponderada= {h_pond:.4f} bits")
print(f"   Ganancia de Info  = {ig:.4f} bits  mayor es mejor")

   Entropía padre    = 0.9764 bits
   Entropía izq (F)  = 0.7642 bits
   Entropía der (M)  = 0.7230 bits
   Entropía ponderada= 0.7380 bits
   Ganancia de Info  = 0.2384 bits  mayor es mejor


La entropía del nodo padre fue 0.9764 bits, casi el máximo teórico de 1 bit para clasificación binaria, lo que refleja máxima incertidumbre. Después del split, la entropía ponderada descendió a 0.7335 bits, logrando una ganancia de información de 0.2429 bits; es decir saber el sexo del pasajero reduce la incertidumbre sobre su supervivencia en casi un cuarto de bit.

## **NOTA**

***Bits*** usan logaritmo base 2. Un bit es la cantidad de información que obtienes al saber el resultado de un volado: cara o cruz, dos opciones equiprobables.

***Nats*** usan logaritmo natural (base e). Es la misma cantidad de información pero expresada en una escala diferente.


1 bit=ln⁡(2)≈0.693 nats

Si la entropía de un nodo es 0.9764 bits, en nats es 0.9764 × 0.693 = 0.6766 nats, es el mismo valor de incertidumbre = solo cambia la escala. En clasificación, sklearn calcula la entropía en bits y el log loss en nats pero como la conversión entre ellos es multiplicar por una constante, el orden de los splits candidatos nunca cambia y el árbol resultante es idéntico.

In [24]:
# LOG LOSS
# El árbol después del split predice una probabilidad constante por hoja: si eres mujer, tu probabilidad de sobrevivir es p_left
# si eres hombre, p_right
# np.where asigna esa probabilidad a cada pasajero según su sexo
# np.column_stack arma la matriz de probabilidades que espera log_loss: columna 0 = P(no sobrevive), columna 1 = P(sobrevive)
# Se compara contra ll_base, que es el log loss sin hacer ningún split (prediciendo la media global para todos).
n_total = len(y_arr)
p_left  = y_left.mean()
p_right = y_right.mean()

p_pred        = np.where(sex_arr == 0, p_left, p_right)
p_pred_matrix = np.column_stack([1 - p_pred, p_pred])

ll      = log_loss(y_arr, p_pred_matrix)
p_base  = y_arr.mean()
ll_base = log_loss(y_arr, np.full((len(y_arr), 2), [1-p_base, p_base]))

# Conversión: entropía ponderada (bits) × ln(2) = log loss (nats)
ll_desde_entropia = h_pond * np.log(2)
# Ganancia en nats = IG en bits × ln(2)
ig_nats = ig * np.log(2)

print(f"   Log Loss (sin split)           = {ll_base:.4f} nats")
print(f"   Log Loss (con split, sklearn)  = {ll:.4f} nats")
print(f"   Mejora en Log Loss             = {ll_base - ll:.4f} nats  ← mayor es mejor")
print(f"   Entropía ponderada × ln(2)     = {ll_desde_entropia:.4f} nats")
print(f"   Log Loss (con split, sklearn)  = {ll:.4f} nats")
print(f"   Diferencia numérica            = {abs(ll - ll_desde_entropia):.2e}  (casi cero)")
print(f"   IG en bits × ln(2)             = {ig_nats:.4f} nats")
print(f"   Mejora en Log Loss             = {ll_base - ll:.4f} nats")

   Log Loss (sin split)           = 0.6768 nats
   Log Loss (con split, sklearn)  = 0.5115 nats
   Mejora en Log Loss             = 0.1653 nats  ← mayor es mejor
   Entropía ponderada × ln(2)     = 0.5115 nats
   Log Loss (con split, sklearn)  = 0.5115 nats
   Diferencia numérica            = 1.11e-16  (casi cero)
   IG en bits × ln(2)             = 0.1653 nats
   Mejora en Log Loss             = 0.1653 nats


Sin ningún split, predecir la probabilidad media global para todos los pasajeros generó un Log Loss de 0.6768 nats. Al dividir por sexo, el modelo predice 0.7778 de probabilidad de sobrevivir para mujeres y 0.2005 para hombres, reduciendo el Log Loss a 0.5115 nats. La mejora absoluta fue de 0.1653 nats, esto demuestra que multiplicar la entropía ponderada por ln(2) da exactamente 0.5115 nats, igual al log loss de sklearn, con una diferencia numérica de 1.11e-16, que es esencialmente cero (error de punto flotante de la máquina). Esto demuestra empíricamente que ambos criterios son idénticos.

In [25]:
# ÁRBOLES COMPLETOS CON CADA CRITERIO
# Se entrena un árbol con cada uno de los tres criterios
# max_depth=3 limita la profundidad para que el árbol no haga overfitting y sea visualizable
# Para cada árbol se guardan el modelo, su accuracy y su log loss en el diccionario resultados
# aunque el criterio interno de split puede ser cualquiera de los tres, siempre se evalúa con log loss al final
criterios  = ['gini', 'entropy', 'log_loss']
resultados = {}

for criterio in criterios:
    clf = DecisionTreeClassifier(criterion=criterio, max_depth=3, random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)
    acc    = accuracy_score(y_test, y_pred)
    ll_val = log_loss(y_test, y_prob)

    resultados[criterio] = {'model': clf, 'acc': acc, 'logloss': ll_val}
    print(f"\n  Criterio : {criterio.upper()}")
    print(f"  Accuracy  = {acc:.4f}")
    print(f"  Log Loss  = {ll_val:.4f}")


  Criterio : GINI
  Accuracy  = 0.7413
  Log Loss  = 0.7462

  Criterio : ENTROPY
  Accuracy  = 0.7413
  Log Loss  = 0.7456

  Criterio : LOG_LOSS
  Accuracy  = 0.7413
  Log Loss  = 0.7456


Los tres criterios obtienen exactamente 74.13% de accuracy, lo que indica que con una profundidad máxima de 3 los tres convergen a una capacidad predictiva similar en este dataset. Las diferencias en cómo miden la impureza no cambian sustancialmente qué tan bien clasifican. ENTROPY y LOG_LOSS tienen exactamente el mismo Log Loss de evaluación (0.7456), confirmando que producen el mismo árbol; GINI obtiene un Log Loss mayor (0.7462), esto indica que los árboles construidos con entropía generan ligeramente mejores probabilidades.

## Referencias

Analytics Vidhya. (2024, noviembre 18). *Splitting decision trees with Gini impurity*. https://www.analyticsvidhya.com/articles/gini-impurity/

Breiman, L., Friedman, J. H., Olshen, R. A., & Stone, C. J. (1984). *Classification and regression trees*. Chapman & Hall/CRC. https://www.researchgate.net/profile/Dan-Steinberg/publication/265031802_Chapter_10_CART_Classification_and_Regression_Trees/links/567dcf8408ae051f9ae493fe/Chapter-10-CART-Classification-and-Regression-Trees.pdf

Brownlee, J. (2025, noviembre 20). *From Shannon to modern AI: A complete information theory guide for machine learning*. Machine Learning Mastery. https://machinelearningmastery.com/from-shannon-to-modern-ai-a-complete-information-theory-guide-for-machine-learning/

Coralogix. (2025, junio 3). *Understanding binary cross-entropy and log loss for effective model monitoring*. https://coralogix.com/ai-blog/understanding-binary-cross-entropy-and-log-loss-for-effective-model-monitoring/

DataCamp. (2026, febrero 27). *Cross-entropy loss function in machine learning: Enhancing model accuracy*. https://www.datacamp.com/tutorial/the-cross-entropy-loss-function-in-machine-learning

Last9. (2025, julio 10). *What is log loss and cross-entropy*. https://last9.io/blog/understanding-log-loss-and-cross-entropy/

LearnDataSci. (s.f.). *Gini impurity*. https://www.learndatasci.com/glossary/gini-impurity/

ML Cheatsheet Contributors. (s.f.). *Loss functions — ML glossary documentation*. Read the Docs. https://ml-cheatsheet.readthedocs.io/en/latest/loss_functions.html

ScienceDirect Topics. (s.f.). *Information gain — overview*. Elsevier. https://www.sciencedirect.com/topics/computer-science/information-gain

scikit-learn Developers. (2024). *1.10. Decision trees — mathematical formulation*. https://scikit-learn.org/stable/modules/tree.html

scikit-learn Developers. (2024). *DecisionTreeClassifier — scikit-learn 1.8.0 documentation*. https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

Wang, Y., Song, C., & Xia, S.-T. (2015). *Unifying the split criteria of decision trees using Tsallis entropy*. arXiv. https://arxiv.org/pdf/1511.08136